# Feature Engineering for Time-Series Forecasting
> **Project:** AI-Based Product Demand Forecasting System  
> **Objective:** Create lag features, rolling statistics, and calendar indicators
> that capture temporal patterns for ML-based demand forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')

PROCESSED_PATH = '../notebook/data/processed_data.csv'
FEATURES_PATH  = '../notebook/data/features_data.csv'

df = pd.read_csv(PROCESSED_PATH, parse_dates=['InvoiceDate'])
print(f'Loaded processed data: {df.shape}')
df.head(3)

In [ ]:
# ── Aggregate to daily demand (UK market focus) ─────────────────
df_uk = df[df['Country'] == 'United Kingdom'].copy()

daily = (
    df_uk.groupby(df_uk['InvoiceDate'].dt.date)
    .agg(TotalQuantity=('Quantity', 'sum'),
         TotalRevenue=('Revenue', 'sum'),
         NumTransactions=('InvoiceNo', 'nunique'),
         AvgUnitPrice=('UnitPrice', 'mean'))
    .reset_index()
    .rename(columns={'InvoiceDate': 'Date'})
)
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.set_index('Date').asfreq('D').fillna(0).reset_index()

print(f'Daily time-series shape: {daily.shape}')
daily.head()

In [ ]:
# ── Lag features ────────────────────────────────────────────────
LAG_PERIODS = [1, 7, 14, 30]
for lag in LAG_PERIODS:
    daily[f'Lag_{lag}'] = daily['TotalQuantity'].shift(lag)

print('Lag feature columns:', [f'Lag_{l}' for l in LAG_PERIODS])
daily[['Date', 'TotalQuantity'] + [f'Lag_{l}' for l in LAG_PERIODS]].head(35).tail(5)

In [ ]:
# ── Rolling window statistics ────────────────────────────────────
WINDOWS = [7, 14, 30]
for w in WINDOWS:
    daily[f'RolMean_{w}'] = daily['TotalQuantity'].shift(1).rolling(w).mean()
    daily[f'RolStd_{w}']  = daily['TotalQuantity'].shift(1).rolling(w).std()
    daily[f'RolMax_{w}']  = daily['TotalQuantity'].shift(1).rolling(w).max()
    daily[f'RolMin_{w}']  = daily['TotalQuantity'].shift(1).rolling(w).min()

roll_cols = [c for c in daily.columns if c.startswith('Rol')]
print(f'Rolling feature count: {len(roll_cols)}')
print(roll_cols)

In [ ]:
# ── Calendar / seasonal indicators ──────────────────────────────
daily['DayOfWeek']    = daily['Date'].dt.dayofweek
daily['Month']        = daily['Date'].dt.month
daily['Quarter']      = daily['Date'].dt.quarter
daily['WeekOfYear']   = daily['Date'].dt.isocalendar().week.astype(int)
daily['IsWeekend']    = daily['DayOfWeek'].isin([5, 6]).astype(int)
daily['IsMonthStart'] = daily['Date'].dt.is_month_start.astype(int)
daily['IsMonthEnd']   = daily['Date'].dt.is_month_end.astype(int)

# Holiday season flag (Nov–Dec)
daily['IsHolidaySeason'] = daily['Month'].isin([11, 12]).astype(int)

# Fourier terms for weekly and monthly seasonality
daily['SinMonth'] = np.sin(2 * np.pi * daily['Month'] / 12)
daily['CosMonth'] = np.cos(2 * np.pi * daily['Month'] / 12)
daily['SinDOW']   = np.sin(2 * np.pi * daily['DayOfWeek'] / 7)
daily['CosDOW']   = np.cos(2 * np.pi * daily['DayOfWeek'] / 7)

print('Calendar features added.')
daily[['Date','DayOfWeek','Month','IsWeekend','IsHolidaySeason','SinMonth','CosMonth']].head(3)

In [ ]:
# ── Interaction features ────────────────────────────────────────
daily['Lag1_x_RolMean7']   = daily['Lag_1'] * daily['RolMean_7']
daily['Lag7_x_IsHoliday']  = daily['Lag_7'] * daily['IsHolidaySeason']
daily['PriceQty_ratio']    = daily['TotalRevenue'] / (daily['TotalQuantity'] + 1)

print('Interaction features added.')
print('Total columns so far:', daily.shape[1])

In [ ]:
# ── Drop rows with NaN from lag/rolling ─────────────────────────
daily_clean = daily.dropna().reset_index(drop=True)
print(f'Rows after dropping NaN (from lags): {len(daily_clean)}')

# Correlation heatmap of key features
feature_cols = ['TotalQuantity','Lag_1','Lag_7','Lag_14','Lag_30',
                'RolMean_7','RolMean_14','RolMean_30','RolStd_7',
                'IsWeekend','IsHolidaySeason','Month']

corr_matrix = daily_clean[feature_cols].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance via Random Forest (quick proxy) ──────────
TARGET = 'TotalQuantity'
FEATURE_COLS = [c for c in daily_clean.columns
                if c not in [TARGET, 'Date', 'TotalRevenue']]

X = daily_clean[FEATURE_COLS].fillna(0)
y = daily_clean[TARGET]

rf_quick = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_quick.fit(X, y)

importances = pd.Series(rf_quick.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True).tail(15)

importances.plot(kind='barh', color='darkcyan', edgecolor='black')
plt.title('Top-15 Feature Importances (Random Forest Proxy)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise lag features vs target ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, lag in enumerate(LAG_PERIODS):
    axes[i].scatter(daily_clean[f'Lag_{lag}'], daily_clean[TARGET],
                    alpha=0.3, s=8, color='steelblue')
    axes[i].set_title(f'Lag {lag} vs Target')
    axes[i].set_xlabel(f'Lag_{lag}')
    axes[i].set_ylabel('TotalQuantity')

plt.suptitle('Lag Features vs Daily Demand', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final feature set summary ────────────────────────────────────
print('=== Final Feature Set ===')
print(f'Rows   : {daily_clean.shape[0]}')
print(f'Columns: {daily_clean.shape[1]}')
print('\nColumn list:')
for col in daily_clean.columns:
    print(f'  {col:<30} dtype: {daily_clean[col].dtype}')

In [ ]:
# ── Save feature-engineered dataset ─────────────────────────────
os.makedirs(os.path.dirname(FEATURES_PATH), exist_ok=True)
daily_clean.to_csv(FEATURES_PATH, index=False)
print(f'Feature data saved to: {FEATURES_PATH}')